# RoadVision — Training + Inference (YOLO)

Public Score - 0.55276
Private Score - 0.55117

**From EDA:** 810 train / 327 test images, all 1280×720, 13 classes (0–12), 10,475 boxes, dual Tesla T4.

This notebook is end-to-end (one *Run All* → `submission.csv`):
1. Convert `train.csv` → Ultralytics YOLO layout (one `labels/<stem>.txt` per image) + `data.yaml`
2. Leak-resistant train/val split
3. Fine-tune a COCO-pretrained YOLO (transfer learning)
4. Inference on test → packed `PredictionString` → `submission.csv`

### The math, briefly
- **IoU** = area(overlap)/area(union). A prediction is a true positive iff IoU ≥ 0.5 with a same-class GT box.
- **mAP@0.5**: per class, sort predictions by confidence, build the precision–recall curve, AP = area under it; mAP = mean AP over the 13 classes. *Confidence only affects ranking → its absolute scale (0–1 vs 0–100) doesn't change the score.*
- **Transfer learning**: COCO-pretrained weights already encode edges→wheels→vehicle shapes; with only ~810 images we fine-tune rather than learn filters from scratch.
- **Mosaic aug**: stitches 4 images into one, multiplying object/context variety — critical for a small dataset.

> **Before *Save Version → Save & Run All*:** Settings → **Accelerator = GPU** and **Internet = On** (the pip install + COCO weights need it). `submission.csv` is written to `/kaggle/working/`, so it appears in the version's **Output** to submit.
>
> If the install / weight download fails, Internet is off — tell me and I'll send the offline setup (wheel + weights as a Kaggle dataset).

In [1]:
# --- Install / verify Ultralytics ---
try:
    import ultralytics
except ImportError:
    import subprocess
    subprocess.run(["pip", "install", "-q", "ultralytics"], check=True)
    import ultralytics

ultralytics.checks()   # prints versions + detected GPU
from ultralytics import YOLO

Ultralytics 8.4.75 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
Setup complete ✅ (4 CPUs, 31.3 GB RAM, 6960.3/8062.4 GB disk)


In [2]:
import os, re, glob, shutil
from pathlib import Path
import numpy as np
import pandas as pd

In [3]:
# --- Auto-discover dataset paths (same robust logic as the EDA notebook) ---
INPUT_ROOT = Path("/kaggle/input")

TRAIN_CSV  = next(iter(INPUT_ROOT.rglob("train.csv")), None)
SAMPLE_SUB = next(iter(INPUT_ROOT.rglob("sample_submission.csv")), None)

image_dirs = [p for p in INPUT_ROOT.rglob("images") if p.is_dir()]
TRAIN_IMAGES = next((p for p in image_dirs if "train" in str(p).lower()), None)
TEST_IMAGES  = next((p for p in image_dirs if "test"  in str(p).lower()), None)

print("TRAIN_CSV   :", TRAIN_CSV)
print("TRAIN_IMAGES:", TRAIN_IMAGES)
print("TEST_IMAGES :", TEST_IMAGES)
print("SAMPLE_SUB  :", SAMPLE_SUB)
assert TRAIN_CSV and TRAIN_IMAGES and TEST_IMAGES, "Path discovery failed - check /kaggle/input"  

TRAIN_CSV   : /kaggle/input/competitions/road-vision/RoadVision_DUET/train/train.csv
TRAIN_IMAGES: /kaggle/input/competitions/road-vision/RoadVision_DUET/train/images
TEST_IMAGES : /kaggle/input/competitions/road-vision/RoadVision_DUET/test/images
SAMPLE_SUB  : /kaggle/input/competitions/road-vision/RoadVision_DUET/test/sample_submission.csv


In [4]:
# ============================ CONFIG ============================
# Column names (verified in EDA)
COL_IMAGE, COL_CLASS = "image_id", "class_id"
COL_XC, COL_YC, COL_W, COL_H = "x_center", "y_center", "width", "height"

NUM_CLASSES = 13
# Submission uses numeric class IDs, so names here are cosmetic (don't affect score).
# Replace with the real 13 names from the competition page if you want readable plots.
CLASS_NAMES = [str(i) for i in range(NUM_CLASSES)]

# Split
SPLIT_MODE = "temporal_tail"   # "temporal_tail" (leak-safe) | "group_clip" | "random"
VAL_FRAC   = 0.12
SEED       = 42

# Training (start fast; bump IMGSZ->1280 / MODEL->yolo11m if time/quota allows)
MODEL  = "yolo11s.pt"
IMGSZ  = 960
EPOCHS = 60
BATCH  = 16
DEVICE = 0                     # single T4. Dual: [0,1] is ~2x but DDP can be flaky in notebooks.

# Inference (low conf threshold maximizes mAP recall on the PR curve)
PRED_CONF, PRED_IOU, MAX_DET = 0.001, 0.6, 300

# Output dirs
# DATASET in /kaggle/temp -> scratch, NOT saved into the committed Output (keeps it clean,
# and avoids saving ~713 symlinked images). runs + submission.csv stay in /kaggle/working
# so they appear in the version Output (submit the csv; reuse best.pt later if you want).
DATASET = Path("/kaggle/temp/roadvision_yolo")
PROJECT, RUN = "/kaggle/working/runs", "exp"
SUB_PATH = "/kaggle/working/submission.csv"
# ===============================================================
print("config set")

config set


In [5]:
# --- Index image files by stem (filename without extension) ---
def index_images(folder):
    idx = {}
    for e in ("*.jpg", "*.jpeg", "*.png", "*.JPG", "*.JPEG", "*.PNG"):
        for p in folder.rglob(e):
            idx[p.stem] = p          # stem has NO extension; matches image_id minus ".txt"
    return idx

train_idx = index_images(TRAIN_IMAGES)
test_idx  = index_images(TEST_IMAGES)
print("indexed train images:", len(train_idx))
print("indexed test  images:", len(test_idx))

df = pd.read_csv(TRAIN_CSV)
print("annotation rows:", len(df), "| unique image_ids:", df[COL_IMAGE].nunique())

# Keep only annotated images whose image file actually exists
df["stem"] = df[COL_IMAGE].map(lambda s: Path(str(s)).stem)   # strips ".txt"
have = set(train_idx)
missing = sorted(set(df["stem"]) - have)
print("annotated stems with no image file:", len(missing), missing[:3])
df = df[df["stem"].isin(have)].reset_index(drop=True)

indexed train images: 810
indexed test  images: 327
annotation rows: 10475 | unique image_ids: 810
annotated stems with no image file: 0 []


In [6]:
# --- Build the train/val split ---
rng = np.random.default_rng(SEED)

def clip_and_frame(stem):
    m = re.match(r"(.+?)_(\d+)$", stem)            # CCTV01^...120000 , 62
    return (m.group(1), int(m.group(2))) if m else (stem, -1)

stems = sorted(df["stem"].unique())
val_set = set()

if SPLIT_MODE == "temporal_tail":
    # Within each clip, hold out the LAST VAL_FRAC of frames (a contiguous tail).
    # -> train/val frames are temporally separated => almost no adjacent-frame leakage,
    #    and every camera is represented in both splits.
    by_clip = {}
    for s in stems:
        c, f = clip_and_frame(s)
        by_clip.setdefault(c, []).append((f, s))
    for c, items in by_clip.items():
        items.sort()                                # by frame number
        k = max(1, int(round(len(items) * VAL_FRAC)))
        val_set.update(s for _, s in items[-k:])
elif SPLIT_MODE == "group_clip":
    # Hold out whole clips (tests cross-camera generalization; higher variance with 4 clips).
    clips = sorted({clip_and_frame(s)[0] for s in stems})
    rng.shuffle(clips)
    k = max(1, int(round(len(clips) * VAL_FRAC)))
    val_clips = set(clips[:k])
    val_set = {s for s in stems if clip_and_frame(s)[0] in val_clips}
else:  # "random" (NOT recommended: adjacent frames leak across splits)
    perm = rng.permutation(stems)
    k = int(round(len(stems) * VAL_FRAC))
    val_set = set(perm[:k])

train_stems = [s for s in stems if s not in val_set]
val_stems   = [s for s in stems if s in val_set]
print(f"split={SPLIT_MODE}  train={len(train_stems)}  val={len(val_stems)}")

# Per-clip val coverage sanity check
from collections import Counter
print("val images per clip:", dict(Counter(clip_and_frame(s)[0] for s in val_stems)))

split=temporal_tail  train=713  val=97
val images per clip: {'CCTV01^20260318-110000-20260318-120000': 49, 'CCTV02^20260511-140000-20260511-142440': 17, 'CCTV07^20260116-060000-20260116-065959': 11, 'CCTV10^20251112-170000-20251112-175959': 20}


## Why this split
Frames in each clip are ~1–2 apart, so neighbours are near-identical. A **random** split would put frame *N* in train and *N+1* in val → the model "sees" val during training and the local mAP is inflated. The **temporal tail** keeps train and val frames in separate time windows within every camera, giving a more honest estimate while still training on all 4 scenes.

In [7]:
# --- Materialize the Ultralytics dataset (symlink images, write label .txt files) ---
if DATASET.exists():
    shutil.rmtree(DATASET)
for split in ("train", "val"):
    (DATASET / "images" / split).mkdir(parents=True, exist_ok=True)
    (DATASET / "labels" / split).mkdir(parents=True, exist_ok=True)

# Group annotations by image once (fast)
groups = {k: g for k, g in df.groupby("stem")}

def write_split(stem_list, split):
    n_lbl = 0
    for stem in stem_list:
        src = train_idx[stem]
        dst_img = DATASET / "images" / split / (stem + src.suffix)
        try:
            os.symlink(src.resolve(), dst_img)      # symlink = no copy, instant
        except (OSError, FileExistsError):
            if not dst_img.exists():
                shutil.copy(src, dst_img)           # fallback if symlinks disabled
        g = groups[stem]
        lines = [f"{int(r[COL_CLASS])} {r[COL_XC]:.6f} {r[COL_YC]:.6f} {r[COL_W]:.6f} {r[COL_H]:.6f}"
                 for _, r in g.iterrows()]
        (DATASET / "labels" / split / (stem + ".txt")).write_text("\n".join(lines))
        n_lbl += len(lines)
    return n_lbl

n_tr = write_split(train_stems, "train")
n_va = write_split(val_stems, "val")
print(f"wrote labels -> train boxes: {n_tr}  val boxes: {n_va}")

# data.yaml
yaml_text = (
    f"path: {DATASET}\n"
    f"train: images/train\n"
    f"val: images/val\n"
    f"nc: {NUM_CLASSES}\n"
    f"names: {CLASS_NAMES}\n"
)
(DATASET / "data.yaml").write_text(yaml_text)
print("\ndata.yaml:\n" + yaml_text)

# Verify one example label matches the CSV
ex = train_stems[0]
print("example label", ex + ".txt", "->")
print((DATASET / "labels" / "train" / (ex + ".txt")).read_text()[:200])

wrote labels -> train boxes: 9346  val boxes: 1129

data.yaml:
path: /kaggle/temp/roadvision_yolo
train: images/train
val: images/val
nc: 13
names: ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12']

example label CCTV01^20260318-110000-20260318-120000_000062.txt ->
0 0.274617 0.725167 0.032062 0.076556
2 0.376758 0.945028 0.056797 0.070028
3 0.526539 0.871660 0.059547 0.200486
3 0.693270 0.559861 0.079695 0.091194
3 0.695102 0.485764 0.046719 0.057000
10 0.67174


## What Ultralytics does during training
YOLO11 is **anchor-free**: the head predicts, per grid cell, an objectness/class score and box offsets. The loss is **box (CIoU) + classification (BCE) + DFL** (distribution focal loss for sub-pixel box edges). It runs mosaic/affine/HSV augmentation on the fly and disables mosaic for the final epochs (`close_mosaic`) so the model fine-tunes on clean images before stopping.

In [8]:
# --- Train ---
model = YOLO(MODEL)   # downloads COCO-pretrained weights (needs internet on)
results = model.train(
    data=str(DATASET / "data.yaml"),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    device=DEVICE,
    project=PROJECT,
    name=RUN,
    seed=SEED,
    patience=20,          # early stop if val mAP plateaus
    mosaic=1.0,
    close_mosaic=10,      # turn off mosaic for the last 10 epochs
    cos_lr=True,
    verbose=True,
)
print("\nbest weights:", Path(results.save_dir) / "weights" / "best.pt")

Ultralytics 8.4.75 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/kaggle/temp/roadvision_yolo/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=60, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=960, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=exp, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=2

In [9]:
# --- Validate (local mAP estimate) ---
metrics = model.val()
print(f"mAP@0.5      : {metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95 : {metrics.box.map:.4f}")
# Per-class AP@0.5 (spot weak classes)
try:
    for i, ap in zip(metrics.box.ap_class_index, metrics.box.ap50):
        print(f"  class {int(i):2d} ({CLASS_NAMES[int(i)]}): AP50={ap:.3f}")
except Exception as e:
    print("per-class AP unavailable:", e)

Ultralytics 8.4.75 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
YOLO11s summary (fused): 101 layers, 9,417,831 parameters, 0 gradients, 21.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 440.6±255.8 MB/s, size: 434.8 KB)
val: Scanning /kaggle/temp/roadvision_yolo/labels/val.cache... 97 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 97/97 37.0Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 7/7 1.9it/s 3.7s
                   all         97       1129      0.396       0.58      0.492       0.31
                     0         33         91      0.323      0.385      0.295      0.109
                     1         56        145      0.324      0.366      0.314      0.125
                     2         75        262      0.639      0.756      0.789      0.489
                     3         87        284       0.35       0.69      0.336      0.166
                     4         28       

## Inference → submission
For each test image, output every detection (low `conf` → better mAP recall) as
`class conf x_center y_center width height`, space-separated, grouped into one `PredictionString` per image.
`image_id` is the stem + `.txt` (matching the sample). Confidence is scaled to 0–100 to mirror the sample (the metric ranks by it, so the scale itself is harmless).

In [10]:
# --- Predict on test set and build submission.csv ---
best_pt = Path(results.save_dir) / "weights" / "best.pt"
pred_model = YOLO(str(best_pt))

test_paths = sorted(set(str(p) for p in test_idx.values()))
print("test images to predict:", len(test_paths))

preds = pred_model.predict(
    source=test_paths, imgsz=IMGSZ, conf=PRED_CONF, iou=PRED_IOU,
    max_det=MAX_DET, stream=True, verbose=False, device=DEVICE,
)

rows, n_with = [], 0
for r in preds:
    stem = Path(r.path).stem
    b = r.boxes
    parts = []
    if b is not None and len(b) > 0:
        cls   = b.cls.cpu().numpy().astype(int)
        conf  = b.conf.cpu().numpy()
        xywhn = b.xywhn.cpu().numpy()            # normalized xc, yc, w, h
        for i in conf.argsort()[::-1]:           # confidence-descending
            xc, yc, w, h = xywhn[i]
            parts += [str(int(cls[i])), f"{conf[i] * 100:.2f}",
                      f"{xc:.6f}", f"{yc:.6f}", f"{w:.6f}", f"{h:.6f}"]
        n_with += 1
    rows.append({"image_id": stem + ".txt", "PredictionString": " ".join(parts)})

sub = pd.DataFrame(rows).sort_values("image_id").reset_index(drop=True)
sub.to_csv(SUB_PATH, index=False)
print(f"saved {SUB_PATH}  | rows: {len(sub)}  | images with >=1 detection: {n_with}")

test images to predict: 327
saved /kaggle/working/submission.csv  | rows: 327  | images with >=1 detection: 327


In [11]:
# --- Sanity-check the submission against the sample format ---
sub = pd.read_csv(SUB_PATH)
print("rows:", len(sub), "(expected", len(test_idx), ")")
print("columns:", list(sub.columns))
toks = str(sub["PredictionString"].iloc[0]).split()
print("row 0:", len(toks), "tokens ->", len(toks) // 6, "detections; first 6:", toks[:6])
display(sub.head())

if SAMPLE_SUB:
    samp = pd.read_csv(SAMPLE_SUB)
    samp.columns = samp.columns.str.strip()
    print("\nsample columns:", list(samp.columns), "| our columns match:", list(sub.columns) == list(samp.columns))

assert os.path.exists(SUB_PATH), "submission.csv missing!"
print("\nOK -> submission at", SUB_PATH, "(", round(os.path.getsize(SUB_PATH) / 1e6, 2), "MB )")
print("After Save & Run All this file appears in the version Output -> submit it.")

rows: 327 (expected 327 )
columns: ['image_id', 'PredictionString']
row 0: 540 tokens -> 90 detections; first 6: ['10', '82.04', '0.775960', '0.728653', '0.142004', '0.339950']


,image_id,PredictionString
0,CCTV01^20260318-110000-20260318-120000_000061.txt,10 82.04 0.775960 0.728653 0.142004 0.339950 3...
1,CCTV01^20260318-110000-20260318-120000_000065.txt,3 83.79 0.730444 0.630495 0.058347 0.100286 0 ...
2,CCTV01^20260318-110000-20260318-120000_000069.txt,10 80.68 0.810572 0.802967 0.143842 0.385825 4...
3,CCTV01^20260318-110000-20260318-120000_000073.txt,3 81.20 0.257988 0.696271 0.069783 0.111257 3 ...
4,CCTV01^20260318-110000-20260318-120000_000077.txt,4 78.76 0.926609 0.738289 0.076800 0.128654 3 ...



sample columns: ['image_id', 'PredictionString'] | our columns match: True

OK -> submission at /kaggle/working/submission.csv ( 1.14 MB )
After Save & Run All this file appears in the version Output -> submit it.


## If time/quota remains — ways to push mAP
- **Higher resolution**: `IMGSZ=1280` (these are 1280×720; helps the many small/distant vehicles most).
- **Bigger model**: `yolo11m.pt` (more capacity; slower).
- **TTA at inference**: `pred_model.predict(..., augment=True)`.
- **Longer training**: raise `EPOCHS`; `patience` will stop early if it plateaus.
- **Pseudo-labeling**: predict on the competition's unannotated images, keep high-confidence boxes, add to training (semi-supervised).
- **Class imbalance**: from the EDA bar chart, weak/rare classes hurt mAP most — inspect their per-class AP50 above.

**Submit:** *Save Version → Save & Run All*, then attach `submission.csv` from the version's Output (or submit it directly from the competition page).